# Validate the CDC flow (bronze → silver_2)

Run this **after** `01_generate_test_bronze_records` and a pipeline update of `flight_data_prep_silver_pipeline`.

Checks (structured, never a raw crash — each row reports PASS/FAIL with evidence):

1. Row-count reconciliation per table: `bronze` vs `silver_2` counts must match (inserts applied).
2. Test records present in silver: all test IDs (`A9001/A9002`, `P9001/P9002/P9003`, `B95001…B95008`).
3. Gold enrichment: new bookings appear in `gold_booking_insights` with joined airport/passenger attributes (new-airport rows must NOT be 'Unknown Airport').
4. Type conversion: `amount` DECIMAL(10,2), `booking_date` DATE in silver.

In [0]:
from pyspark.sql import functions as F

BRONZE = "claudecatalog.bronze"
SILVER = "claudecatalog.silver_2"

EXPECTED_TEST_IDS = {
    "silver_airports": ["A9001", "A9002"],
    "silver_passengers": ["P9001", "P9002", "P9003"],
    "silver_bookings": [f"B9500{i}" for i in range(1, 9)],
}

checks = []  # (check, detail, expected, actual, status)

def record(check, detail, expected, actual, ok):
    checks.append((check, detail, str(expected), str(actual), "PASS" if ok else "FAIL"))

# --- Check 1: row-count reconciliation bronze vs silver ---
for bronze_tbl, silver_tbl in [
    ("bookings", "silver_bookings"),
    ("passengers", "silver_passengers"),
    ("airports", "silver_airports"),
]:
    try:
        b = spark.table(f"{BRONZE}.{bronze_tbl}").count()
        s = spark.table(f"{SILVER}.{silver_tbl}").count()
        record("row_count_reconciliation", silver_tbl, b, s, b == s)
    except Exception as e:
        record("row_count_reconciliation", silver_tbl, "query ok", f"{type(e).__name__}: {e}", False)

In [0]:
# --- Check 2: test records present in silver ---
key_by_table = {
    "silver_airports": "airport_id",
    "silver_passengers": "passenger_id",
    "silver_bookings": "booking_id",
}

for silver_tbl, ids in EXPECTED_TEST_IDS.items():
    try:
        key = key_by_table[silver_tbl]
        found = (
            spark.table(f"{SILVER}.{silver_tbl}")
            .filter(F.col(key).isin(ids))
            .select(key)
            .distinct()
            .collect()
        )
        found_ids = sorted(r[key] for r in found)
        record("test_ids_in_silver", silver_tbl, sorted(ids), found_ids, found_ids == sorted(ids))
    except Exception as e:
        record("test_ids_in_silver", silver_tbl, "query ok", f"{type(e).__name__}: {e}", False)

# --- Check 4: type conversions in silver_bookings ---
try:
    bad_types = spark.table(f"{SILVER}.silver_bookings").filter(
        ~(
            (F.col("amount").cast("decimal(10,2)") == F.col("amount"))
            & F.col("booking_date").isNotNull()
        )
    ).count()
    record("silver_type_integrity", "amount decimal(10,2), booking_date not null date", 0, bad_types, bad_types == 0)
except Exception as e:
    record("silver_type_integrity", "silver_bookings", "query ok", f"{type(e).__name__}: {e}", False)

In [0]:
# --- Check 3: gold enrichment for the new bookings ---
try:
    gold_new = (
        spark.table(f"{SILVER}.gold_booking_insights")
        .filter(F.col("booking_id").isin(EXPECTED_TEST_IDS["silver_bookings"]))
    )
    gold_rows = gold_new.count()
    record("gold_enrichment", "new bookings in gold_booking_insights",
           len(EXPECTED_TEST_IDS["silver_bookings"]), gold_rows,
           gold_rows == len(EXPECTED_TEST_IDS["silver_bookings"]))
    # bookings at the NEW airports A9001/A9002 must carry the real airport attributes
    new_airport_bookings = gold_new.filter(F.col("airport_id").isin("A9001", "A9002"))
    enriched_correctly = new_airport_bookings.filter(
        (F.col("airport_name") != "Unknown Airport") & (F.col("country").isin("Fictionland", "Testland"))
    ).count()
    record("gold_enrichment", "bookings at A9001/A9002 enriched with real attributes",
           new_airport_bookings.count(), enriched_correctly,
           new_airport_bookings.count() > 0 and enriched_correctly == new_airport_bookings.count())
except Exception as e:
    record("gold_enrichment", "gold_booking_insights", "query ok", f"{type(e).__name__}: {e}", False)

# --- Report ---
report = spark.createDataFrame(
    checks, schema="check STRING, detail STRING, expected STRING, actual STRING, status STRING")
display(report)

failed = [c for c in checks if c[4] == "FAIL"]
if failed:
    print(f"CDC flow validation FAILED for {len(failed)}/{len(checks)} checks. "
          "If the pipeline update has not run yet, run it first and re-run this notebook.")
else:
    print(f"CDC flow validation PASSED: {len(checks)}/{len(checks)} checks. "
          "Incremental CDC load (bronze Change Data Feed -> silver_2) is working end-to-end.")